# HMC Crown vigilance — Head A-vig, frozen REVE-base (experimental)

**Public retrain path:** same HF windows as kernel 10 (`crown2_vigilance_hmc` / `crown4_vigilance_hmc`) +
`scripts/train_hmc_crown_vig_reve.py`. Shipped heads are **heads-only** in [`neurofeed_heads`](https://github.com/windwerfer/neurofeed_heads).

**Bring your own gated REVE-base weights** (and positions bank). Do **not** redistribute REVE-base in public
repos, Kaggle public datasets, or HF. Set local paths via env / `data/models/reve-base` (never commit weights).

Montages **crown2_strong** (C=2) vs **crown4_hmc** (C=4). Ship bar: test macro-F1 ≥ 0.60.

**Maintainer note:** Private Kaggle scratch OK to leave private. ISRUC / L-FAME / LUNA / SEED-VIG stay out.


In [ ]:
import os, sys, json, shutil
from pathlib import Path

CANDIDATE_ROOTS = [
    Path('/kaggle/working/neurofeed_train'),
    Path('.').resolve(),
    Path('/kaggle/working'),
]
ROOT = None
for cand in CANDIDATE_ROOTS:
    if (cand / 'scripts' / 'train_hmc_crown_vig_reve.py').exists():
        ROOT = cand
        break
if ROOT is None:
    ROOT = Path('/kaggle/working/neurofeed_train')
    ROOT.mkdir(parents=True, exist_ok=True)
    print('WARN: train script not found yet; clone/copy neurofeed_train into', ROOT)

WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.').resolve()
INPUT = Path('/kaggle/input')
print('ROOT', ROOT)
print('inputs', sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else None)


In [ ]:
from pathlib import Path
import os, shutil, sys

def first(*cands):
    for c in cands:
        if c is None:
            continue
        p = Path(c)
        if p.exists():
            return p
    return None

HF_ROOT = first(
    os.environ.get('HF_WINDOWS_ROOT'),
    os.environ.get('DATASETS_ROOT'),
    ROOT / 'data' / 'hf_windows',
    Path('./data/hf_windows'),
)
HF_MAP = {
    'vigilance_hmc_crown2': 'crown2_vigilance_hmc',
    'vigilance_hmc_crown4': 'crown4_vigilance_hmc',
}
CROWN = first(
    '/kaggle/input/muse-eeg-heads-crown-vig',
    '/kaggle/input/datasets/windwerfer/muse-eeg-heads-crown-vig',
)

(ROOT / 'datasets').mkdir(parents=True, exist_ok=True)
for pack, hf_cfg in HF_MAP.items():
    dest = ROOT / 'datasets' / pack
    dest.mkdir(parents=True, exist_ok=True)
    win_dest = dest / 'windows'
    src = None
    if HF_ROOT:
        for cand in [HF_ROOT / hf_cfg / 'windows', HF_ROOT / hf_cfg]:
            if cand.exists():
                src = cand if cand.name == 'windows' else (cand / 'windows' if (cand / 'windows').exists() else cand)
                break
    if src is None and CROWN:
        for cand in [CROWN / 'datasets' / pack / 'windows', CROWN / 'datasets' / pack]:
            if cand.exists():
                src = cand if cand.name == 'windows' else (cand / 'windows' if (cand / 'windows').exists() else cand)
                break
    if src is None:
        print(f'MISSING windows for {pack} ({hf_cfg}). Download HF config and set HF_WINDOWS_ROOT.')
        continue
    if win_dest.exists() and any(win_dest.glob('*_windows.npz')):
        print(pack, 'already staged nights', len(list(win_dest.glob('*_windows.npz'))))
        continue
    if win_dest.exists():
        shutil.rmtree(win_dest)
    if src.name == 'windows':
        shutil.copytree(src, win_dest)
    else:
        win_dest.mkdir(parents=True, exist_ok=True)
        for f in src.glob('*_windows.npz'):
            shutil.copy2(f, win_dest / f.name)
    print(pack, 'nights', len(list(win_dest.glob('*_windows.npz'))), 'from', src)

# REVE-base: bring-your-own gated weights. Prefer env / local data/; optional private scratch only.
reve = first(
    os.environ.get('REVE_BASE_DIR'),
    ROOT / 'data' / 'models' / 'reve-base',
    CROWN / 'models' / 'reve-base' if CROWN else None,
)
pos = first(
    os.environ.get('REVE_POSITIONS_DIR'),
    ROOT / 'data' / 'models' / 'reve-positions',
    CROWN / 'models' / 'reve-positions' if CROWN else None,
)
assert reve and Path(reve).is_dir(), (
    'REVE-base missing. Bring your own gated weights (do not redistribute). '
    'Set REVE_BASE_DIR or place under data/models/reve-base/.'
)
assert pos and Path(pos).is_dir(), (
    'REVE positions bank missing. Set REVE_POSITIONS_DIR or place under data/models/reve-positions/.'
)
# Stage where FrozenREVEEncoder looks for optional local prefer
cache_models = ROOT / 'kaggle_datasets' / 'muse-eeg-heads-cache' / 'models'
cache_models.mkdir(parents=True, exist_ok=True)
for name, src in (('reve-base', Path(reve)), ('reve-positions', Path(pos))):
    dst = cache_models / name
    if dst.resolve() != src.resolve():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    print(name, 'ready', dst)

# Offline einops if vendored next to private scratch (optional)
if CROWN and (CROWN / 'vendor').is_dir():
    sys.path.insert(0, str(CROWN / 'vendor'))


In [ ]:
import torch, sys, os
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
from src.reve_encoder import FrozenREVEEncoder
enc = FrozenREVEEncoder(source_sr=256.0, channel_names=['C3','C4'], prefer_local=True, device='cpu')
notes = enc.adapter_notes()
print('local model', notes.get('model_from_local'), 'positions', notes.get('positions_from_local'))
y = enc(torch.zeros(1, 2, 512))
print('smoke', tuple(y.shape))
del enc


In [ ]:
import runpy, os, sys
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
out = ROOT / 'exports' / 'hmc_crown_vig_reve'
sys.argv = ['train_hmc_crown_vig_reve.py', '--out', str(out)]
runpy.run_path(str(ROOT / 'scripts' / 'train_hmc_crown_vig_reve.py'), run_name='__main__')


In [ ]:
import json, shutil
from pathlib import Path
cmp_path = ROOT / 'exports' / 'hmc_crown_vig_reve' / 'compare.json'
cmp = json.loads(cmp_path.read_text())
summary = {
    'any_ship_candidate': cmp.get('any_ship_candidate'),
    'both_below_bar': cmp.get('both_below_bar'),
    'sampling': cmp.get('sampling'),
    'f1': cmp.get('f1'),
}
print(json.dumps(summary, indent=2))
(WORKING / 'compare_summary.json').write_text(json.dumps(summary, indent=2))
shutil.copy2(cmp_path, WORKING / 'compare.json')
